# ⚡ StreamDiffusion Colab Server for TouchDesigner
This notebook runs **StreamDiffusion (SD-Turbo)** accelerated on Google Colab Pro GPU and bridges real-time streaming directly to your local **TouchDesigner** via a secure Cloudflare Tunnel.

### Instructions:
1. Make sure your runtime has a GPU: Go to **Runtime -> Change runtime type -> Hardware accelerator: GPU** (Select **A100**, **L4**, or **T4**).
2. Click **Runtime -> Run all** (or run each cell sequentially).
3. Cell 4 will warm up the pipeline and print your **TouchDesigner WebSocket URL** (`wss://....trycloudflare.com/ws/stream`).

In [ ]:
# Step 1: Check GPU
!nvidia-smi

In [ ]:
# Step 2: Clone StreamDiffusion & Install Compatible Dependencies
import os
print('Setting up StreamDiffusion environment...')

# Clone repo if not already present
if not os.path.exists('StreamDiffusion'):
    !git clone https://github.com/cumulo-autumn/StreamDiffusion.git

%cd StreamDiffusion
!pip install -q -e .
# Pin compatible versions and remove conflicting peft
!pip install -q 'transformers<4.40.0' 'huggingface-hub<0.26.0'
!pip uninstall -y peft || true
!pip install -q diffusers accelerate fastapi uvicorn websockets python-multipart opencv-python-headless pillow

# Download cloudflared for tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print('✅ Setup complete!')

In [ ]:
# Step 3: Write server.py
server_code = "import os\nimport sys\nimport traceback\n\n# 1. Compatibility shims for huggingface_hub\nimport huggingface_hub\nimport huggingface_hub.constants\n\nif not hasattr(huggingface_hub.constants, 'hf_cache_home'):\n    huggingface_hub.constants.hf_cache_home = getattr(\n        huggingface_hub.constants, 'HF_HOME', os.path.expanduser('~/.cache/huggingface')\n    )\n\nif not hasattr(huggingface_hub, 'HfFolder'):\n    class HfFolder:\n        @classmethod\n        def get_token(cls):\n            try:\n                from huggingface_hub import get_token\n                return get_token()\n            except Exception:\n                return None\n        @classmethod\n        def save_token(cls, token):\n            pass\n    huggingface_hub.HfFolder = HfFolder\n\nif not hasattr(huggingface_hub, 'cached_download'):\n    from huggingface_hub import hf_hub_download\n    huggingface_hub.cached_download = hf_hub_download\n\n# 2. Compatibility shims for transformers\nimport transformers\nimport transformers.utils\n\nif not hasattr(transformers.utils, 'FLAX_WEIGHTS_NAME'):\n    transformers.utils.FLAX_WEIGHTS_NAME = 'flax_model.msgpack'\nif not hasattr(transformers.utils, 'FLAX_WEIGHTS_INDEX_NAME'):\n    transformers.utils.FLAX_WEIGHTS_INDEX_NAME = 'flax_model.msgpack.index.json'\n\nif not hasattr(transformers, 'EncoderDecoderCache'):\n    class EncoderDecoderCache:\n        pass\n    transformers.EncoderDecoderCache = EncoderDecoderCache\n    try:\n        import transformers.cache_utils\n        transformers.cache_utils.EncoderDecoderCache = EncoderDecoderCache\n    except Exception:\n        pass\n\nimport time\nimport asyncio\nimport json\nimport logging\nimport cv2\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom fastapi import FastAPI, WebSocket, WebSocketDisconnect\nfrom fastapi.responses import HTMLResponse\n\n# Disable autograd globally for fast inference\ntorch.set_grad_enabled(False)\n\nsys.path.append('.')\nfrom utils.wrapper import StreamDiffusionWrapper\nfrom streamdiffusion.image_utils import postprocess_image\n\n# Patch image_utils.pt_to_numpy just in case\ntry:\n    import streamdiffusion.image_utils\n    orig_pt_to_numpy = streamdiffusion.image_utils.pt_to_numpy\n    def safe_pt_to_numpy(images):\n        return images.detach().cpu().permute(0, 2, 3, 1).float().numpy()\n    streamdiffusion.image_utils.pt_to_numpy = safe_pt_to_numpy\nexcept Exception:\n    pass\n\nlogging.basicConfig(level=logging.INFO)\nlogger = logging.getLogger(\"StreamDiffusionServer\")\napp = FastAPI()\n\nstream_txt2img = None\nstream_img2img = None\npipeline_ready = False\ncurrent_prompt = \"cinematic cyberpunk cityscape, neon lights, 8k, detailed\"\ncurrent_mode = \"txt2img\"\ncurrent_seed = 42\ncurrent_strength = 0.6\n\ndef set_seed(s):\n    torch.manual_seed(s)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(s)\n\ndef apply_prompt(new_prompt):\n    global current_prompt, stream_txt2img, stream_img2img\n    current_prompt = new_prompt\n    try:\n        if stream_txt2img is not None:\n            if hasattr(stream_txt2img, \"update_prompt\"):\n                stream_txt2img.update_prompt(new_prompt)\n            elif hasattr(stream_txt2img.stream, \"update_prompt\"):\n                stream_txt2img.stream.update_prompt(new_prompt)\n        if stream_img2img is not None:\n            if hasattr(stream_img2img, \"update_prompt\"):\n                stream_img2img.update_prompt(new_prompt)\n            elif hasattr(stream_img2img.stream, \"update_prompt\"):\n                stream_img2img.stream.update_prompt(new_prompt)\n        logger.info(f\"Updated prompt to: {new_prompt}\")\n    except Exception as e:\n        logger.error(f\"Error updating prompt: {e}\")\n\ndef apply_strength(new_strength):\n    global current_strength, stream_img2img\n    current_strength = float(new_strength)\n    try:\n        if stream_img2img is not None:\n            if hasattr(stream_img2img, \"update_strength\"):\n                stream_img2img.update_strength(current_strength)\n            elif hasattr(stream_img2img, \"stream\") and hasattr(stream_img2img.stream, \"t_list\"):\n                t0 = max(5, min(44, int(45 - current_strength * 40)))\n                stream_img2img.stream.t_list = [t0, 45]\n                stream_img2img.prepare(prompt=current_prompt, num_inference_steps=50)\n                logger.info(f\"Updated img2img denoise strength to {current_strength:.2f} (t_index_list={[t0, 45]})\")\n    except Exception as e:\n        logger.error(f\"Error updating strength: {e}\")\n\ndef run_model_inference(input_image=None):\n    \"\"\"\n    Runs StreamDiffusion inference under torch.no_grad() inside the executor thread.\n    Never use torch.inference_mode() because StreamDiffusion caches tensors between steps.\n    \"\"\"\n    with torch.no_grad():\n        if current_mode in (\"img2img\", \"v2v\") and input_image is not None and stream_img2img is not None:\n            return stream_img2img(input_image)\n        if stream_txt2img is not None:\n            return stream_txt2img()\n        return None\n\ndef compress_frame(arr_bgr, max_bytes=48000):\n    \"\"\"\n    Dynamically compresses JPEG to guarantee payload stays strictly under TouchDesigner's\n    64KB WebSocket limit (typically around 25KB-42KB).\n    \"\"\"\n    for q in (60, 50, 40, 30):\n        ret, buf = cv2.imencode('.jpg', arr_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), q])\n        if ret and len(buf) <= max_bytes:\n            return buf.tobytes()\n    h, w = arr_bgr.shape[:2]\n    small = cv2.resize(arr_bgr, (int(w * 0.8), int(h * 0.8)), interpolation=cv2.INTER_AREA)\n    for q in (45, 30, 20):\n        ret, buf = cv2.imencode('.jpg', small, [int(cv2.IMWRITE_JPEG_QUALITY), q])\n        if ret and len(buf) <= max_bytes:\n            return buf.tobytes()\n    return buf.tobytes()\n\ndef load_pipeline():\n    global stream_txt2img, stream_img2img, pipeline_ready\n    device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    print(f\">>> Initializing StreamDiffusion (SD-Turbo) on {device.upper()}...\")\n\n    # 1. Text-to-Image pipeline: [0, 16, 32, 45] produces full, coherent, prompt-matching frames\n    print(\">>> Initializing Text-to-Image pipeline (t_index_list=[0, 16, 32, 45])...\")\n    stream_txt2img = StreamDiffusionWrapper(\n        model_id_or_path=\"stabilityai/sd-turbo\",\n        t_index_list=[0, 16, 32, 45],\n        frame_buffer_size=1,\n        width=512,\n        height=512,\n        warmup=5,\n        acceleration=\"none\",\n        mode=\"txt2img\",\n        use_lcm_lora=False,\n        use_tiny_vae=True,\n        cfg_type=\"none\",\n        use_denoising_batch=True,\n        device=device,\n        seed=current_seed,\n    )\n    set_seed(current_seed)\n\n    # 2. Image-to-Image / StreamV2V pipeline: initial t_index_list based on current_strength\n    t0_init = max(5, min(44, int(45 - current_strength * 40)))\n    print(f\">>> Initializing Image-to-Image / StreamV2V pipeline (t_index_list=[{t0_init}, 45], strength={current_strength:.2f})...\")\n    try:\n        from streamdiffusion import StreamDiffusion\n        core_pipe = stream_txt2img.stream.pipe\n        stream_core = StreamDiffusion(\n            pipe=core_pipe,\n            t_index_list=[t0_init, 45],\n            torch_dtype=core_pipe.dtype,\n            width=512,\n            height=512,\n            frame_buffer_size=1,\n            use_denoising_batch=True,\n            cfg_type=\"none\",\n        )\n        if hasattr(stream_txt2img.stream, \"vae\") and stream_txt2img.stream.vae:\n            stream_core.vae = stream_txt2img.stream.vae\n\n        class SharedImg2ImgWrapper:\n            def __init__(self, core):\n                self.stream = core\n            def prepare(self, prompt, **kwargs):\n                self.stream.prepare(prompt, **kwargs)\n            def update_prompt(self, prompt):\n                self.stream.update_prompt(prompt)\n            def update_strength(self, strength: float):\n                # Dynamically maps strength [0.1..1.0] to start timestep index [44..5]\n                t0 = max(5, min(44, int(45 - strength * 40)))\n                new_t_list = [t0, 45]\n                if getattr(self.stream, \"t_list\", None) != new_t_list:\n                    self.stream.t_list = new_t_list\n                    self.stream.prepare(prompt=current_prompt, num_inference_steps=50)\n                    logger.info(f\"Updated img2img denoise strength to {strength:.2f} (t_index_list={new_t_list})\")\n            def __call__(self, image):\n                # When passed a PIL Image, stream(...) automatically preprocesses it\n                # without diffusers value-range warnings!\n                x_output = self.stream(image)\n                pil_out = postprocess_image(x_output, output_type=\"pil\")\n                return pil_out[0] if isinstance(pil_out, list) else pil_out\n\n        stream_img2img = SharedImg2ImgWrapper(stream_core)\n    except Exception as e:\n        print(\">>> Fallback to standalone img2img wrapper:\", e)\n        stream_img2img = StreamDiffusionWrapper(\n            model_id_or_path=\"stabilityai/sd-turbo\",\n            t_index_list=[t0_init, 45],\n            frame_buffer_size=1,\n            width=512,\n            height=512,\n            warmup=5,\n            acceleration=\"none\",\n            mode=\"img2img\",\n            use_lcm_lora=False,\n            use_tiny_vae=True,\n            cfg_type=\"none\",\n            use_denoising_batch=True,\n            device=device,\n            seed=current_seed,\n        )\n\n    print(\">>> Preparing both pipelines with initial prompt...\")\n    stream_txt2img.prepare(prompt=current_prompt, num_inference_steps=50)\n    stream_img2img.prepare(prompt=current_prompt, num_inference_steps=50)\n\n    print(\">>> Warming up models...\")\n    with torch.no_grad():\n        for _ in range(4):\n            stream_txt2img()\n        dummy_img = Image.new(\"RGB\", (512, 512), color=(128, 128, 128))\n        for _ in range(3):\n            stream_img2img(dummy_img)\n\n    pipeline_ready = True\n    print(\">>> StreamDiffusion pipeline is READY! <<<\", flush=True)\n\n@app.on_event(\"startup\")\nasync def startup_event():\n    loop = asyncio.get_event_loop()\n    await loop.run_in_executor(None, load_pipeline)\n\n@app.get(\"/\")\ndef index():\n    return HTMLResponse(\"\"\"\n    <html>\n        <head><title>StreamDiffusion TouchDesigner Server</title></head>\n        <body style=\"font-family: sans-serif; padding: 40px; background: #121212; color: #fff;\">\n            <h2>StreamDiffusion (SD-Turbo) Colab Server</h2>\n            <p>Status: <span style=\"color: #0f0;\">Running</span></p>\n            <p>Connect your TouchDesigner client via the WebSocket endpoint: <code>/ws/stream</code></p>\n        </body>\n    </html>\n    \"\"\")\n\n@app.websocket(\"/ws/stream\")\nasync def websocket_stream_endpoint(websocket: WebSocket):\n    global current_prompt, current_mode, current_seed, current_strength\n    await websocket.accept()\n    logger.info(\"TouchDesigner client connected to WebSocket.\")\n\n    await websocket.send_text(json.dumps({\n        \"type\": \"welcome\",\n        \"message\": \"StreamDiffusion server connected\",\n        \"prompt\": current_prompt,\n        \"mode\": current_mode,\n        \"seed\": current_seed,\n        \"strength\": current_strength,\n    }))\n\n    CHUNK_SIZE = 2048\n    frame_counter = 0\n\n    async def send_chunked_frame(ws: WebSocket, jpg_bytes: bytes):\n        nonlocal frame_counter\n        frame_counter = (frame_counter + 1) % 256\n        total_chunks = (len(jpg_bytes) + CHUNK_SIZE - 1) // CHUNK_SIZE\n        for chunk_idx in range(total_chunks):\n            start = chunk_idx * CHUNK_SIZE\n            end = min(start + CHUNK_SIZE, len(jpg_bytes))\n            chunk = jpg_bytes[start:end]\n            header = bytes([frame_counter, chunk_idx, total_chunks])\n            try:\n                await ws.send_bytes(header + chunk)\n            except Exception:\n                break\n\n    txt2img_task = None\n    stop_event = asyncio.Event()\n    img2img_counter = 0\n    td_chunk_buffers = {}\n\n    async def txt2img_loop():\n        target_fps = 25\n        frame_delay = 1.0 / target_fps\n        frame_count = 0\n        logger.info(f\"txt2img_loop started! (pipeline_ready={pipeline_ready}, current_mode={current_mode})\")\n        while not stop_event.is_set():\n            start_time = time.time()\n            if pipeline_ready and current_mode == \"txt2img\":\n                try:\n                    loop = asyncio.get_event_loop()\n                    t0 = time.time()\n                    output_tensor = await loop.run_in_executor(None, run_model_inference)\n                    inf_ms = (time.time() - t0) * 1000\n                    \n                    if output_tensor is not None:\n                        if isinstance(output_tensor, Image.Image):\n                            arr_bgr = cv2.cvtColor(np.array(output_tensor), cv2.COLOR_RGB2BGR)\n                        elif hasattr(output_tensor, \"cpu\"):\n                            arr = (output_tensor.detach().squeeze().cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)\n                            arr_bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)\n                        else:\n                            arr_bgr = np.array(output_tensor)\n\n                        jpg_bytes = compress_frame(arr_bgr)\n                        await send_chunked_frame(websocket, jpg_bytes)\n                        \n                        frame_count += 1\n                        if frame_count % 10 == 0:\n                            try:\n                                await websocket.send_text(json.dumps({\n                                    \"type\": \"stats\",\n                                    \"inf_ms\": round(inf_ms, 1),\n                                    \"mode\": current_mode,\n                                    \"strength\": current_strength\n                                }))\n                            except Exception:\n                                pass\n                                \n                        if frame_count % 25 == 1:\n                            logger.info(f\"StreamDiffusion active: frame #{frame_count} ({inf_ms:.1f}ms, {len(jpg_bytes)} bytes)\")\n                except Exception as e:\n                    if not stop_event.is_set():\n                        logger.error(f\"Inference error in txt2img: {traceback.format_exc()}\")\n\n            elapsed = time.time() - start_time\n            delay = max(0.005, frame_delay - elapsed)\n            await asyncio.sleep(delay)\n\n    txt2img_task = asyncio.create_task(txt2img_loop())\n\n    try:\n        while True:\n            message = await websocket.receive()\n            if message.get(\"type\") == \"websocket.disconnect\":\n                logger.info(\"TouchDesigner client disconnected normally.\")\n                break\n\n            if \"text\" in message:\n                try:\n                    data = json.loads(message[\"text\"])\n                    msg_type = data.get(\"type\", \"\")\n\n                    if msg_type == \"control\" or \"prompt\" in data or \"mode\" in data or \"seed\" in data or \"strength\" in data:\n                        new_prompt = data.get(\"prompt\")\n                        new_mode = data.get(\"mode\")\n                        new_seed = data.get(\"seed\")\n                        new_strength = data.get(\"strength\")\n\n                        if new_prompt is not None and (new_prompt != current_prompt or data.get(\"force\", False)):\n                            apply_prompt(new_prompt)\n\n                        if new_mode is not None:\n                            mode_str = str(new_mode).lower()\n                            if \"txt\" in mode_str or \"text\" in mode_str:\n                                current_mode = \"txt2img\"\n                            elif \"img\" in mode_str or \"webcam\" in mode_str or \"v2v\" in mode_str or \"noise\" in mode_str:\n                                current_mode = \"img2img\"\n                            else:\n                                current_mode = \"img2img\"\n                            logger.info(f\"Switching mode to: {current_mode}\")\n\n                        if new_seed is not None and new_seed != current_seed:\n                            current_seed = new_seed\n                            set_seed(current_seed)\n\n                        if new_strength is not None and abs(float(new_strength) - current_strength) > 0.01:\n                            apply_strength(float(new_strength))\n\n                    elif msg_type == \"ping\":\n                        await websocket.send_text(json.dumps({\"type\": \"pong\", \"time\": time.time()}))\n\n                except Exception as e:\n                    logger.error(f\"Error handling text message: {e}\")\n\n            elif \"bytes\" in message:\n                if pipeline_ready and current_mode in (\"img2img\", \"v2v\"):\n                    raw = message[\"bytes\"]\n                    try:\n                        frame_bytes = None\n                        # Reassemble chunked frames from TouchDesigner:\n                        # Header: b\"FRM:\" + frame_id + chunk_idx + total_chunks (7 bytes)\n                        if raw.startswith(b\"FRM:\") and len(raw) >= 7:\n                            frame_id = raw[4]\n                            chunk_idx = raw[5]\n                            total_chunks = raw[6]\n                            chunk_data = raw[7:]\n\n                            if frame_id not in td_chunk_buffers:\n                                td_chunk_buffers.clear()\n                                td_chunk_buffers[frame_id] = [None] * total_chunks\n\n                            if chunk_idx < total_chunks:\n                                td_chunk_buffers[frame_id][chunk_idx] = chunk_data\n\n                            if all(c is not None for c in td_chunk_buffers[frame_id]):\n                                frame_bytes = b\"\".join(td_chunk_buffers[frame_id])\n                                del td_chunk_buffers[frame_id]\n                        elif raw.startswith(b\"\\xff\\xd8\"):\n                            frame_bytes = raw\n                        else:\n                            frame_bytes = raw\n\n                        if frame_bytes is not None:\n                            nparr = np.frombuffer(frame_bytes, np.uint8)\n                            in_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)\n                            if in_bgr is not None:\n                                in_rgb = cv2.cvtColor(in_bgr, cv2.COLOR_BGR2RGB)\n                                in_pil = Image.fromarray(in_rgb).resize((512, 512))\n\n                                loop = asyncio.get_event_loop()\n                                t0 = time.time()\n                                output = await loop.run_in_executor(None, run_model_inference, in_pil)\n                                inf_ms = (time.time() - t0) * 1000\n\n                                if isinstance(output, Image.Image):\n                                    out_bgr = cv2.cvtColor(np.array(output), cv2.COLOR_RGB2BGR)\n                                elif hasattr(output, \"cpu\"):\n                                    out_arr = (output.detach().squeeze().cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)\n                                    out_bgr = cv2.cvtColor(out_arr, cv2.COLOR_RGB2BGR)\n                                else:\n                                    out_bgr = np.array(output)\n\n                                out_bytes = compress_frame(out_bgr)\n                                await send_chunked_frame(websocket, out_bytes)\n\n                                img2img_counter += 1\n                                if img2img_counter % 10 == 0:\n                                    try:\n                                        await websocket.send_text(json.dumps({\n                                            \"type\": \"stats\",\n                                            \"inf_ms\": round(inf_ms, 1),\n                                            \"mode\": current_mode,\n                                            \"strength\": current_strength\n                                        }))\n                                    except Exception:\n                                        pass\n\n                                if img2img_counter % 25 == 1:\n                                    logger.info(f\"StreamDiffusion img2img: frame #{img2img_counter} ({inf_ms:.1f}ms, {len(out_bytes)} bytes)\")\n                    except Exception as e:\n                        if not stop_event.is_set():\n                            logger.error(f\"Error in img2img processing: {traceback.format_exc()}\")\n\n    except WebSocketDisconnect:\n        logger.info(\"TouchDesigner client disconnected gracefully.\")\n    except Exception as e:\n        logger.error(f\"WebSocket error: {e}\")\n    finally:\n        stop_event.set()\n        if txt2img_task:\n            txt2img_task.cancel()\n            try:\n                await txt2img_task\n            except asyncio.CancelledError:\n                pass\n\nif __name__ == \"__main__\":\n    import uvicorn\n    uvicorn.run(app, host=\"0.0.0.0\", port=8000, log_level=\"info\")\n"

with open('server.py', 'w') as f:
    f.write(server_code.strip())
print('✅ server.py written successfully!')

In [ ]:
# Step 4: Launch Server, Warm Up & Start Cloudflare Tunnel
import os
import subprocess
import time
import sys
import re
from IPython.display import display, HTML

# Kill any lingering process on port 8000
os.system('fuser -k 8000/tcp 2>/dev/null || true')

print('Starting Python server and downloading SD-Turbo weights (~2.5GB)...')
server_proc = subprocess.Popen(
    [sys.executable, 'server.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Stream logs until model is fully ready
ready = False
for line in server_proc.stdout:
    print(line, end='', flush=True)
    if 'READY!' in line or 'Application startup complete' in line or 'Uvicorn running' in line:
        ready = True
        break
    if server_proc.poll() is not None:
        break

if ready:
    print('\nStarting Cloudflare Tunnel to expose port 8000...')
    tunnel_proc = subprocess.Popen(
        ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    tunnel_url = None
    for line in tunnel_proc.stdout:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break

    if tunnel_url:
        ws_url = tunnel_url.replace('https://', 'wss://') + '/ws/stream'
        display(HTML(f'''
        <div style="background:#1a2e1a; border:2px solid #00ff88; padding:20px; border-radius:10px; margin:20px 0;">
            <h2 style="color:#00ff88; margin:0 0 10px 0;">🚀 StreamDiffusion Server is READY & STREAMING!</h2>
            <p style="color:#eee; font-size:16px; margin:0 0 10px 0;">Copy this URL and paste it into TouchDesigner:</p>
            <div style="background:#000; padding:12px; border-radius:6px; font-family:monospace; font-size:18px; color:#58a6ff;">
                {ws_url}
            </div>
        </div>
        '''))
else:
    print('\n❌ Server failed to start. Review error logs above.')

# Keep server running
try:
    while True:
        line = server_proc.stdout.readline()
        if line:
            print(line, end='', flush=True)
        time.sleep(0.01)
except KeyboardInterrupt:
    print('Stopping server...')
    server_proc.terminate()
    if 'tunnel_proc' in locals():
        tunnel_proc.terminate()
